In [1]:
import time
notebook_start = time.perf_counter()

%pip install -e /home/darshan/A6/PCSAFT_cDFT/thermoift

Obtaining file:///home/darshan/A6/PCSAFT_cDFT/thermoift
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for thermoift (pyproject.toml) ... done
  Created wheel for thermoift: filename=thermoift-0.2.0-0.editable-py3-none-any.whl size=1730 sha256=f7841d1fea3940f9b1db72b712287332391fda92125b51d9b219c5293752aa09
  Stored in directory: /tmp/pip-ephem-wheel-cache-md3nqfqb/wheels/fd/2f/c4/54a2ee5cd16a9bf5b183bbe5c28d1b3ba4926fb0261a13e1e4
Successfully built thermoift
  Attempting uninstall: thermoift
    Found existing installation: thermoift 0.2.0
    Uninstalling thermoift-0.2.0:
      Successfully uninstalled thermoift-0.2.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from thermoift import FeedsBuilder
from pathlib import Path
from itertools import combinations
import pandas as pd, json

OUTPUT_DIR = Path("CSV_feeds")
OUTPUT_DIR.mkdir(exist_ok=True)

COMPONENTS  = ["CO2", "H2", "Ar", "N2", "CH4", "O2", "CO", "H2S"]
CO2_LEVELS  = (0.95, 0.96, 0.97, 0.98, 0.99)
SEED        = 550055

# Dirichlet alpha: based on typical industrial impurity bounds
# Higher alpha = component tends to take larger share of impurity
# Lower alpha = component tends toward trace amounts
ALPHA = {
    "N2":  4.0,   # up to 4% in industrial streams
    "CH4": 4.0,   # up to 5% (dominant impurity)
    "H2":  4.0,   # up to 4%
    "Ar":  4.0,   # moderate
    "O2":  0.1,   # trace (10 ppm)
    "CO":  0.1,   # trace (35 ppm)
    "H2S": 0.1,   # trace (100 ppm)
}

builder = FeedsBuilder(rng_type="PCG64", seed=SEED)

### 1. Random feeds (Dirichlet sampling)

In [ ]:
df_random = builder.generate_random_feeds(
    components=COMPONENTS,
    mixture_sizes=(2, 3),
    co2_levels=CO2_LEVELS,
    n_random_samples=200,
    alpha=ALPHA,
)
print(f"Random feeds: {len(df_random)}")
df_random.head()

### 2. Systematic feeds (grid at fixed step)

In [ ]:
df_systematic = builder.generate_systematic_feeds(
    components=COMPONENTS,
    mixture_sizes=(2, 3),
    co2_levels=CO2_LEVELS,
    step=0.01,
    min_fraction=0.005,  # Each impurity must be at least 0.5% - removes extreme compositions
)
print(f"Systematic feeds: {len(df_systematic)}")
df_systematic.head()

### 3. Industrial feeds (bounds + ranges templates)

In [5]:
builder.load_industrial_templates("IndustrialFeeds.json")

df_ind_bounds = builder.industrial_feeds_from_bounds(
    components=COMPONENTS,
    co2_levels=CO2_LEVELS,
)

df_ind_ranges = builder.industrial_feeds_from_ranges(
    components=COMPONENTS,
)

df_industrial = pd.concat([df_ind_bounds, df_ind_ranges], ignore_index=True)
print(f"Industrial feeds: {len(df_industrial)}  (bounds={len(df_ind_bounds)}, ranges={len(df_ind_ranges)})")
df_industrial.head()

Industrial feeds: 320  (bounds=180, ranges=140)


,CO2,H2,Ar,N2,CH4,O2,CO,H2S,feed_source,template_name,mixture_size,active_components
0,0.95,0.014421,0.001108,0.017433,0.016991,4.259497e-09,0.000005,4.239840e-05,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
1,0.95,0.014304,0.008261,0.027269,0.000153,7.706299e-06,0.000005,2.096971e-07,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
2,0.95,0.024889,0.001037,0.000959,0.023050,9.957225e-06,0.000002,5.403979e-05,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
3,0.95,0.001081,0.014417,0.018579,0.015800,6.792258e-06,0.000017,9.809439e-05,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"
4,0.95,0.003282,0.023445,0.011735,0.011526,1.997905e-06,0.000004,6.273310e-06,industrial_bound,netl_limits,8,"CO2,O2,H2S,N2,CO,H2,Ar,CH4"


### 4. Combined feed (70% random / 20% systematic / 10% industrial)

In [6]:
df_all = builder.combine_feeds_random_systematic_industrial(
    components=COMPONENTS,
    n_total_target=100,
    co2_levels=CO2_LEVELS,
    frac_random=0.70,
    frac_systematic=0.20,
    frac_industrial=0.10,
)

print(f"Total feeds: {len(df_all)}")
print()
print(df_all["feed_source"].value_counts())
print()
df_all.head(10)

Total feeds: 100

feed_source
random              70
systematic          20
industrial_range     6
industrial_bound     4
Name: count, dtype: int64



,feed_source,template_name,mixture_size,active_components,CO2,H2,Ar,N2,CH4,O2,CO,H2S
0,random,dirichlet,3,"CO2,O2,H2S",0.99,0.000000,0.000000,0.000000,0.000000,0.009913,0.000000,0.000087
1,random,dirichlet,4,"CO2,Ar,O2,H2S",0.97,0.000000,0.000655,0.000000,0.000000,0.021311,0.000000,0.008035
2,random,dirichlet,5,"CO2,H2,N2,CH4,H2S",0.98,0.001592,0.000000,0.009957,0.003244,0.000000,0.000000,0.005207
3,random,dirichlet,3,"CO2,N2,O2",0.98,0.000000,0.000000,0.004014,0.000000,0.015986,0.000000,0.000000
4,random,dirichlet,6,"CO2,H2,Ar,CH4,O2,H2S",0.97,0.002582,0.008530,0.000000,0.010223,0.004606,0.000000,0.004060
5,random,dirichlet,5,"CO2,H2,Ar,CO,H2S",0.95,0.034182,0.000576,0.000000,0.000000,0.000000,0.000386,0.014856
6,random,dirichlet,3,"CO2,H2,Ar",0.97,0.000793,0.029207,0.000000,0.000000,0.000000,0.000000,0.000000
7,random,dirichlet,4,"CO2,H2,Ar,H2S",0.96,0.006347,0.025629,0.000000,0.000000,0.000000,0.000000,0.008023
8,random,dirichlet,2,"CO2,N2",0.97,0.000000,0.000000,0.030000,0.000000,0.000000,0.000000,0.000000
9,random,dirichlet,7,"CO2,H2,Ar,N2,O2,CO,H2S",0.98,0.000812,0.003428,0.001605,0.000000,0.005285,0.007627,0.001243


### 5. Save all feeds

In [7]:
# Save individual feeds
df_random.to_csv(OUTPUT_DIR / "Random_compositions.csv", index=False)
df_systematic.to_csv(OUTPUT_DIR / "Systematic_compositions.csv", index=False)
df_industrial.to_csv(OUTPUT_DIR / "Industrial_compositions.csv", index=False)

# Save combined feed
df_all.to_csv(OUTPUT_DIR / "Combined_compositions.csv", index=False)

print(f"Saved to {OUTPUT_DIR}/:")
print(f"  Random_compositions.csv       : {len(df_random)} rows")
print(f"  Systematic_compositions.csv   : {len(df_systematic)} rows")
print(f"  Industrial_compositions.csv   : {len(df_industrial)} rows")
print(f"  Combined_compositions.csv     : {len(df_all)} rows")

Saved to CSV_feeds/:
  Random_compositions.csv       : 28000 rows
  Systematic_compositions.csv   : 455 rows
  Industrial_compositions.csv   : 320 rows
  Combined_compositions.csv     : 100 rows


### 6. Generate default KIJ pairs config

In [ ]:
# Generate default KIJ_pairs.json (all pairs by default set to "zero")

COMPONENTS = ["CO2", "H2", "Ar", "N2", "CH4", "O2", "CO", "H2S"]
pairs = [f"{a}-{b}" for a, b in combinations(COMPONENTS, 2)]
kij_pairs = {pair: "zero" for pair in pairs}

kij_file = OUTPUT_DIR / "KIJ_pairs.json"
with open(kij_file, "w") as f:
    json.dump(kij_pairs, f, indent=2)

print(f"Generated {kij_file} with {len(kij_pairs)} pairs (all set to 'zero')")
print("\nEdit this file to override specific pairs:")
print("  Options: 'zero', 'constant', 'linear', 'quadratic'")

Generated CSV_feeds/KIJ_pairs.json with 28 pairs (all set to 'zero')

Edit this file to override specific pairs:
  Options: 'zero', 'constant', 'linear'


In [9]:
elapsed = time.perf_counter() - notebook_start
print(f"\nNotebook runtime: {elapsed:.1f}s")


Notebook runtime: 4.2s
